In [1]:
import tensorflow as tf
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# TensorFlow Lite 모델 로드
tflite_model_file = '/content/drive/MyDrive/최강<포빅아A4>/AI프로젝트/견인/mobilenetv2rms_quantized.tflite'
interpreter = tf.lite.Interpreter(model_path=tflite_model_file)
interpreter.allocate_tensors()

# 입력 및 출력 텐서 정보 가져오기
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()
input_shape = input_details[0]['shape']

# 데이터셋 로드
test_dir = '/content/drive/MyDrive/최강<포빅아A4>/AI프로젝트/견인/데이터/test'  # 테스트 데이터 디렉토리 경로
batch_size = 32  # 배치 크기

test_dataset = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    shuffle=False,
    image_size=(input_shape[1], input_shape[2]),
    batch_size=batch_size,
    label_mode='int'
)

# 모델 예측 및 평가
predictions = []
labels = []

for images, y_true in test_dataset:
    for i in range(images.shape[0]):
        # 이미지 전처리
        img = np.expand_dims(images[i].numpy(), axis=0).astype(np.float32)

        # 모델 입력 설정
        interpreter.set_tensor(input_details[0]['index'], img)

        # 모델 실행
        interpreter.invoke()

        # 결과 추출
        output_data = interpreter.get_tensor(output_details[0]['index'])
        pred = np.argmax(output_data)

        # 결과 저장
        predictions.append(pred)
        labels.append(y_true[i].numpy())

# 성능 지표 계산
accuracy = accuracy_score(labels, predictions)
conf_matrix = confusion_matrix(labels, predictions)
class_report = classification_report(labels, predictions)

print(f"모델 정확도: {accuracy:.4f}")
print("혼동 행렬:")
print(conf_matrix)
print("분류 보고서:")
print(class_report)


Found 347 files belonging to 4 classes.
모델 정확도: 0.6599
혼동 행렬:
[[ 25   5  22  13]
 [  0 105   5   3]
 [  0  50  45   1]
 [  0  10   9  54]]
분류 보고서:
              precision    recall  f1-score   support

           0       1.00      0.38      0.56        65
           1       0.62      0.93      0.74       113
           2       0.56      0.47      0.51        96
           3       0.76      0.74      0.75        73

    accuracy                           0.66       347
   macro avg       0.73      0.63      0.64       347
weighted avg       0.70      0.66      0.64       347

